In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = '../data/5g_core_traffic_energy_management_dataset.csv'
df = pd.read_csv(file_path)

df['Timestamp'] = pd.to_datetime(df['Timestamp'])
df = df.sort_values('Timestamp').reset_index(drop=True)

print(f"Data spans from {df['Timestamp'].min()} to {df['Timestamp'].max()}")
print(f"Initial shape: {df.shape}")

Data spans from 2024-01-01 00:00:00 to 2024-02-22 01:55:00
Initial shape: (15000, 49)


In [2]:
checks = {
    'Full-row duplicates': int(df.duplicated().sum()),
    'Duplicate Timestamps': int(df['Timestamp'].duplicated().sum()),
    'Time gaps != 5 mins': int((df['Timestamp'].diff().dropna() != pd.Timedelta(minutes=5)).sum()),
    'Negative numeric values': int((df.select_dtypes('number') < 0).sum().sum()),
    'Percentage > 100': int((df[['CPU_Utilization_Pct', 'Memory_Utilization_Pct']] > 100).sum().sum())
}

print("--- Data Integrity Checks ---")
for check_name, result in checks.items():
    print(f"{check_name}: {result}")

--- Data Integrity Checks ---
Full-row duplicates: 0
Duplicate Timestamps: 0
Time gaps != 5 mins: 0
Negative numeric values: 0
Percentage > 100: 0


In [3]:
congestion_cols = ['Network_Condition', 'Congestion_Index', 'QoS_Satisfaction_Score','SLA_Violation_Count', 'Future_Congestion_Index']
future_cols = ['Future_Traffic_Demand_Mbps']
energy_derived_cols = ['Energy_Consumption_kWh', 'Energy_Efficiency_Mbps_per_kWh', 'Carbon_Emission_kg', 'Operational_Cost_USD']
cols_to_drop = congestion_cols + future_cols + energy_derived_cols
df_clean = df.drop(columns=cols_to_drop)

print(f"Dropped {len(cols_to_drop)} columns to prevent leakage and overlap.")
print(f"Remaining features: {df_clean.shape[1]}")

Dropped 10 columns to prevent leakage and overlap.
Remaining features: 39


In [4]:
n = len(df_clean)
i_train = int(0.70 * n)
i_val = int(0.85 * n)

df_clean['split'] = np.where(np.arange(n) < i_train, 'train', np.where(np.arange(n) < i_val, 'val', 'test'))

split_summary = df_clean.groupby('split', sort=False).agg(
    rows=('Timestamp', 'size'),
    start=('Timestamp', 'min'),
    end=('Timestamp', 'max'),
    power_mean=('Total_Power_Watt', 'mean'),
    power_std=('Total_Power_Watt', 'std')
).round(2)
display(split_summary)

df_clean.to_csv('../data/prepared_energy_dataset.csv', index=False)

/var/folders/t9/5qyb5pw554s_w_jt9c13myrh0000gn/T/ipykernel_74016/1822006407.py:13: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  ).round(2)


,rows,start,end,power_mean,power_std
split,,,,,
train,10500,2024-01-01 00:00:00,2024-02-06 10:55:00,1405.12,298.24
val,2250,2024-02-06 11:00:00,2024-02-14 06:25:00,1406.43,299.61
test,2250,2024-02-14 06:30:00,2024-02-22 01:55:00,1414.47,300.01
